# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiezugwu/Data-Science/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np
D=pd.read_parquet("/content/decline_recovery.parquet")
# 1. Earliest 30 days = 90-day total minus the two later windows
D["impressions_first30"] = D.impressions_90d - D.impressions_last_30d - D.impressions_prev_30d

# 2. Early momentum, blanked where the earliest window is under the 50 floor
ok = D.impressions_first30 >= 50
D["early_momentum_pct"] = (D.impressions_prev_30d / D.impressions_first30.where(ok) - 1) * 100

# 3. Prior-window CTR as a ratio of totals, x100 to match the file's convention
okp = D.impressions_prev_30d >= 50
D["prev30_ctr_pct"] = D.clicks_prev_30d / D.impressions_prev_30d.where(okp) * 100

# 4. Log scale for the heavy tail
D["log_prev30_impressions"] = np.log1p(D.impressions_prev_30d)

# 5. Page age when the outcome window began
D["age_at_outcome_start_days"] = D.content_age_days - 30
D.head()

,client_hash_id,content_hash_id,keyword_hash_id,keyword_char_count,keyword_token_count,public_url_hash_id,public_url_char_count,public_url_path_depth,content_title_hash_id,content_title_char_count,...,needs_ctr_fix,needs_engagement_fix,ai_opportunity,is_underperformer,is_declining,impressions_first30,early_momentum_pct,prev30_ctr_pct,log_prev30_impressions,age_at_outcome_start_days
0,client_2094c6eb080311d5,content_9c786aea8fb2f1c6,keyword_d879cf3e66407e25,47,7,url_57e0b96b5c053290,98,3,title_48769559fe2659be,48,...,False,False,False,False,True,0,NaN,0.000000,5.117994,21
1,client_2094c6eb080311d5,content_ea9b68bdccddcd4a,keyword_4000d86bb749226b,36,7,url_2f9bddde85164ed7,87,3,title_9de9bcfe0a9b6af8,36,...,True,False,False,False,True,0,NaN,0.000000,6.502790,21
2,client_2094c6eb080311d5,content_4610b9c0a99de467,keyword_1e5f049eb5b7dff9,42,7,url_65d19d39cf4859db,93,3,title_19ccc000e7a54aa8,43,...,True,False,False,False,True,0,NaN,0.733496,6.016157,21
3,client_2094c6eb080311d5,content_d5ec72bd00a5165f,keyword_b93dd6aa011fdd91,32,7,url_eefa7f3e0a53e404,83,3,title_b44521f255eae17f,33,...,False,False,False,False,True,0,NaN,0.000000,5.416100,21
4,client_2094c6eb080311d5,content_e460345ff721587a,keyword_c8212eedb93ad237,33,6,url_0812e375eec6575c,84,3,title_46fdae840eb1ee6b,34,...,False,False,False,False,True,0,NaN,0.000000,5.459586,21


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
1. impressions_first30-Impressions in the earliest 30 days of the 90-day window. The subtraction creates no blanks (0 negatives across 56,253 rows). 3,746 rows are a real zero, and I kept them as zeros instead of treating them as missing. Yes, it exists. It covers the earliest window, before both the prior and the last 30 days.
2. early_momentum_pct-How much visibility changed from the earliest window to the prior one, which is a pre-decline trend. Blank (NaN) where impressions_first30 is under 50, which is 5,801 rows (10.3%). It is not filled. Either use a model that accepts blanks or add a “had enough early data” flag, but don't use fillna(0). Yes, it exists. Both of its inputs come from windows that end before the outcome window.
3. prev30_ctr_pct- The floor is the same, but in this file it blanks nothing because every row has at least 50 prior-window impressions. Zeros are real: many pages had impressions but no clicks. Yes it exists. It uses only the prior 30 days.
4. log_prev30_impressions- No blanks. The prior-window columns are complete in this file, and log1p also handles zeros. Yes it exists, Same window as above.
5. age_at_outcome_start_days- No negatives.Yes it exists, since the creation date was fixed before the window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# LEAKY 1: the label restated as a feature (the most obvious leak)
import pandas as pd, numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from scipy.stats import spearmanr
R = pd.read_parquet("/content/ranking_lifecycle.parquet")[["content_hash_id", "impressions_30d"]]
D = D.merge(R, on="content_hash_id", how="left")

D["severity"] = -(D.impressions_last_30d / D.impressions_prev_30d - 1) * 100

# five honest features
f = D.impressions_90d - D.impressions_last_30d - D.impressions_prev_30d
D["impressions_first30"] = f
D["early_momentum_pct"] = (D.impressions_prev_30d / f.where(f >= 50) - 1) * 100
D["prev30_ctr_pct"] = D.clicks_prev_30d / D.impressions_prev_30d * 100
D["log_prev30_impressions"] = np.log1p(D.impressions_prev_30d)


# the leaky feature: outcome-window impressions, arriving through the join
D["LEAKY_log_joined_30d"] = np.log1p(D.impressions_30d)

honest = ["impressions_first30", "early_momentum_pct", "prev30_ctr_pct",
          "log_prev30_impressions", "age_at_outcome_start_days"]
leaky = honest + ["LEAKY_log_joined_30d"]

model = make_pipeline(SimpleImputer(strategy="median"),
                      QuantileTransformer(n_quantiles=200),
                      LinearRegression())

severe = (D.severity >= D.severity.quantile(.75)).to_numpy()
def p_at_k(s, k=50): return severe[np.argsort(-np.asarray(s))[:k]].mean()

for name, cols in [("honest (5)", honest), ("honest + LEAKY (6)", leaky)]:
    pred = cross_val_predict(model, D[cols], D.severity, groups=D.client_hash_id, cv=GroupKFold(5))
    print(name, spearmanr(pred, D.severity)[0], p_at_k(pred), p_at_k(pred, 500), severe.mean())



honest (5) 0.21234700731055622 0.48 0.634 0.25001333262226016
honest + LEAKY (6) 0.8373388926231805 1.0 1.0 0.25001333262226016


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
health_score-	Hand-written product score that encodes a decision someone already made, so it's the baseline to beat, never an input.
trend_pct-	The label itself, precomputed.
trend_direction-	A category made from trend_pct.
is_declining-	Derived from the trend, and constant here since every row is declining.
needs_indexing-	Hand-written product flag, so using it learns the old rule, not the world.
is_quick_win-	Hand-written product flag.
needs_ctr_fix-	Hand-written product flag.
needs_engagement_fix-	Hand-written product flag.
ai_opportunity-	Hand-written product flag.
is_underperformer-	Hand-written product flag.
impressions_30d, clicks_30d- Equal to the last-30-day values in decline_recovery, so they are the outcome.
pageviews_30d, sessions_30d, users_30d, engaged_sessions_30d-Traffic measured during the outcome window.
ai_sessions_30d, scroll_events_30d	Same window.
ctr_30d, engagement_rate_30d, scroll_rate_30d, ai_traffic_pct_30d- 	Rates computed over the outcome window.
avg_position_30d-	Position in the outcome window, and zeros may mean “no data”.
impression_tier, position_tier-	Buckets built from outcome-window values.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.